In [ ]:
from pathlib import Path
import os

def find_repo_root(start: Path) -> Path:
    for parent in [start] + list(start.parents):
        if (parent / 'pyproject.toml').exists():
            return parent
    return start

REPO_ROOT = find_repo_root(Path.cwd())

if 'NEPTUNE_API_TOKEN' not in os.environ:
    os.environ.setdefault('NEPTUNE_MODE', 'offline')

os.chdir(REPO_ROOT)


In [ ]:
from loguru import logger
from transformers import BartForConditionalGeneration, BertForSequenceClassification
from os.path import join


from carl.inference_components.conditional_low_level_policy import (
    ConditionalLowLevelPolicy,
    TransformerConditionalLowLevelPolicy,
)
from carl.inference_components.subgoal_generator import (
    TransformerSubgoalGenerator,
)
from carl.inference_components.value import TransformerValue, Value


logger.info('testing solve components')

In [ ]:
components_prefix = str(REPO_ROOT / "rl-data/validation/sokoban/components/full_data") + '/'

path_to_policy_weights: str = join(components_prefix, 'policy/checkpoint-94820')
path_to_cllp_weights: str = join(components_prefix, 'cllp/8/checkpoint-167585')
path_to_value_function_weights: str = join(components_prefix, 'value/checkpoint-1343100')
path_to_generator_weights_k8: str = join(components_prefix, 'generator/border/8/checkpoint-75856')
path_to_generator_weights_k4: str = join(components_prefix, 'generator/border/4/checkpoint-75856')

In [ ]:
from carl.environment.sokoban.env import SokobanEnv
from carl.environment.sokoban.tokenizer import SokobanTokenizer
import functools

env_tokenizer: SokobanTokenizer = SokobanTokenizer()
env_class = functools.partial(SokobanEnv, tokenizer=env_tokenizer)
env: SokobanEnv = env_class()

In [ ]:
from transformers import BartConfig, BertConfig
from carl.inference_components.subgoal_generator import AdaptiveSubgoalGenerator

from carl.inference_components.validator import BasicValidator

board_tokens = env.tokenizer.size_of_board[0] * env.tokenizer.size_of_board[1]

subgoal_generation_kwargs: dict[str, int] = {
    'num_beams': 2,
    'num_return_sequences': 1,
    'max_new_tokens': board_tokens + 1,
}

subgoal_generator_cls = functools.partial(
    TransformerSubgoalGenerator, generator_network_class=BartForConditionalGeneration.from_pretrained
)

adaptive_subgoal_generator = AdaptiveSubgoalGenerator(
    generator_k_list=[8, 4],
    subgoal_generator_class=subgoal_generator_cls,
    paths_to_generator_weights=[path_to_generator_weights_k8, path_to_generator_weights_k4],
    env=env,
    subgoal_generation_kwargs=subgoal_generation_kwargs,
)

adaptive_subgoal_generator.construct_network()


value_function: Value = TransformerValue(
    value_network_class=BertForSequenceClassification.from_pretrained,
    path_to_value_network_weights=path_to_value_function_weights,
    env=env,
    type_of_evaluation='regression',
)
value_function.construct_network()

cllp: ConditionalLowLevelPolicy = TransformerConditionalLowLevelPolicy(
    BertForSequenceClassification.from_pretrained, path_to_cllp_weights, env
)
cllp.construct_network()

validator = BasicValidator(env_class(), cllp, budget_for_achieving_subgoal=8)
validator.construct_network()

In [ ]:
import functools
from carl.planners.adasubs import AdasubsPlanner
from carl.solver.subgoal_search import Solver

SOLVER_BUDGET = 20
NUM_BOARDS = 1

AdasubsPlannerCls = functools.partial(AdasubsPlanner, generators_k_list=[8, 4])

solver = Solver(
    SOLVER_BUDGET,
    AdasubsPlannerCls,
    adaptive_subgoal_generator,
    validator,
    value_function,
)

In [ ]:
from carl.environment.instance_generator import (
    BasicInstanceGenerator,
    GeneralIterableDataLoader,
)

path_to_folder_with_data = str(REPO_ROOT / "rl-data/validation/sokoban/progress/boards_1000_b4_gs25_c300_p0.35")

env = SokobanEnv(SokobanTokenizer(None, size_of_board=(12, 12)), num_boxes=4)
instance_generator = BasicInstanceGenerator(
    generator=GeneralIterableDataLoader(path_to_folder_with_data), batch_size=1
)

In [ ]:
initial_state_loader = iter(instance_generator.reset_dataloader())
inputs = []
figs = []
for _ in range(NUM_BOARDS):
    initial_state = next(initial_state_loader).cpu().numpy()[0]
    inputs.append(initial_state)

In [ ]:
# Vizualize inputs
fig = env.many_states_to_repr(inputs, titles=['input_{}'.format(i) for i in range(len(inputs))])
display(fig)

# Sequential Solve

In [ ]:
outputs_sequential = []
for i, input_board in enumerate(inputs):
    print('Solving board number', i)
    output = solver.solve(input_board)
    outputs_sequential.append(output)

In [ ]:
# Logging Function

from carl.planners.base import Experience


def display_solutions(outputs: list[Experience]):
    figs = []
    for j, output in enumerate(outputs):
        if output.solution.solved:
            subgoals = output.solution.subgoal_path
            titles = ['initial_state'] + [
                f'subgoal[{i}] k={output.solution.subgoal_distance_path[i]}'
                for i in range(len(subgoals))
            ]
            
            fig = env.many_states_to_repr(subgoals, titles=titles)
        else:
            fig = env.state_to_repr(initial_state, title='initial_state (unsolved)')

        figs.append(fig)

    for fig in figs:
        display(fig)
        
display_solutions(outputs_sequential)

In [ ]:
if outputs_sequential:
    initial_state = outputs_sequential[0].search_info.search_tree.state
    display(env.state_to_repr(initial_state, title='initial_state'))

if len(outputs_sequential) > 1:
    initial_state2 = outputs_sequential[1].search_info.search_tree.state
    display(env.state_to_repr(initial_state2, title='initial_state2'))

In [ ]:
initial_node = initial_state = outputs_sequential[0].search_info.search_tree
initial_node.children[0].children

In [ ]:
env.many_states_to_repr(
    [initial_node.state, initial_node.children[0].state, initial_node.children[1].state],
    titles=['initial_state', 'child_0', 'child_1'])